# Meeting 10: the class notebook

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 10.

This notebook holds the three Colab blocks of the meeting, about eight minutes each, one for each act. The slides explain the ideas; here your group runs them on its own data and posts what it finds. The **full notebooks** of the three acts, linked on the meeting's page, explain every step and hold the work for after the meeting.

## How this works

- **At the start of the meeting**, open this notebook, choose **Save a copy in Drive** in the **File** menu, run the setup cell, and type your group's number in the cell after it. Then leave the notebook open: the first block comes after the first slides.
- **Work in a group of two or three.** Every member runs their own copy.
- **Three blocks**, one for each act. A block has one or two cells to run, and one thing to change.
- **Three posts** on the discussion board of this meeting, one in each block. **One member posts for the group**, and it need not be the same member each time. **Every post begins with the names of all the members of your group.** The second and the third post are replies to the first. Under every image, write one sentence that says what the image shows.
- **Nothing is uploaded.** The posts are the record of your group's work. **Review Quiz 10** opens when the meeting ends; its questions name the steps of the full notebooks.

<font color="#c00000" size="5"><b>Your turn.</b></font> Double-click this cell and replace this line with the names of the members of your group.

In [ ]:
# Setup. Run this cell first.
import io
import math
import random
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# The address of the course's files for this unit, and of its folders.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
DATA, IMAGES, EMBEDDINGS = SITE + "data/", SITE + "images/", SITE + "embeddings/"


def average_time(function, data, repeats=5):
    """The average time of one call function(data), in seconds, over several calls."""
    total = 0.0
    for _ in range(repeats):
        start = time.perf_counter()
        function(data)
        total += time.perf_counter() - start
    return total / repeats


def read_image(file):
    """Read an image file into a NumPy array with values from 0 to 1."""
    return np.array(Image.open(urlopen(IMAGES + file))) / 255.0


def show(pictures, titles):
    """Draw arrays as pictures: an array with two dimensions in grey, one with three in colour."""
    fig, axes = plt.subplots(1, len(pictures), figsize=(5 * len(pictures), 4.2), squeeze=False)
    for ax, picture, title in zip(axes.flat, pictures, titles):
        ax.imshow(np.clip(picture, 0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


print("Setup is done.")

<font color="#c00000" size="5"><b>Your turn.</b></font> Type your group's number in the next cell, in place of the `0`, and run it. A group without a number draws one: run `random.randint(1, 30)` in a new cell once, and tell the others. The number names your group's dataset, its image and its word.

In [ ]:
GROUP = 0        # replace 0 with your group's number, from 1 to 30

assert 1 <= GROUP <= 30, "Type your group's number in place of the 0."

datasets = pd.read_csv(DATA + "datasets.csv", index_col="number")
images = pd.read_csv(IMAGES + "images.csv", index_col="number")
STARTERS = ["river", "guitar", "doctor", "coffee", "winter", "soccer", "piano", "bridge", "forest", "teacher",
            "ocean", "bread", "train", "garden", "mountain", "engine", "painter", "kitchen", "market", "planet",
            "island", "violin", "tiger", "hospital", "library", "airport", "chocolate", "hurricane", "bicycle", "volcano"]
info = datasets.loc[GROUP]
picture_info = images.loc[GROUP]
WORD = STARTERS[GROUP - 1]
print(f"Group {GROUP}. Dataset: {info['title']}. Image: {picture_info['title']}. Word: '{WORD}'.")

## <font color="#1967d2"><b>Act 1: A table of data is a matrix</b></font>

Your dataset is read with pandas and stored as a matrix `X` with n rows and m columns, and as a list of lists `rows`. The next cell standardizes it in the three ways of the slides, with loops on the lists, with the same loops on the array, and vectorized, and measures each. It takes up to a minute for the largest datasets.

In [ ]:
df = pd.read_csv(DATA + info["file"])
X = df.to_numpy(dtype=float)
n, m = X.shape
rows = X.tolist()
print(f"Dataset {GROUP}: {info['title']}. n = {n:,} rows, m = {m} columns, {n * m:,} numbers.")


def standardize_loops(rows):
    """Version 1: loops on Python lists."""
    n, m = len(rows), len(rows[0])
    means = [sum(row[j] for row in rows) / n for j in range(m)]
    stds = [math.sqrt(sum((row[j] - means[j]) ** 2 for row in rows) / n) for j in range(m)]
    return [[(row[j] - means[j]) / stds[j] for j in range(m)] for row in rows]


def standardize_array_loops(X):
    """Version 2: the same loops, reading a NumPy array one entry at a time."""
    n, m = X.shape
    Z = np.empty((n, m))
    for j in range(m):
        total = 0.0
        for i in range(n):
            total += X[i, j]
        mean = total / n
        total = 0.0
        for i in range(n):
            total += (X[i, j] - mean) ** 2
        std = math.sqrt(total / n)
        for i in range(n):
            Z[i, j] = (X[i, j] - mean) / std
    return Z


def standardize_vectorized(X):
    """Version 3: operations on the whole array; the loop runs inside NumPy."""
    return (X - X.mean(axis=0)) / X.std(axis=0)


times = {
    "lists, loops": average_time(standardize_loops, rows, 3),
    "NumPy array, loops": average_time(standardize_array_loops, X, 3),
    "NumPy array, vectorized": average_time(standardize_vectorized, X, 20),
}
Z = standardize_vectorized(X)

print(f"{'version':26s}{'seconds per call':>18s}{'nanoseconds per number':>25s}")
for version, t in times.items():
    print(f"{version:26s}{t:18.4f}{t / (n * m) * 1e9:25.1f}")

fig, ax = plt.subplots(figsize=(8, 2.8))
bars = ax.barh(list(times), list(times.values()), height=0.5, color="#2a78d6")
ax.bar_label(bars, labels=[f"{t:.4f} s" for t in times.values()], padding=4)
ax.invert_yaxis()
ax.margins(x=0.15)
ax.set_xlabel("average time of one call, in seconds")
ax.set_title(f"Standardizing dataset {GROUP}: {n:,} rows, {m} columns")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

per_number = {version: t / (n * m) * 1e9 for version, t in times.items()}
print("Line for the discussion board:")
print(f"Dataset {GROUP} ({info['title']}), n = {n:,}, m = {m}. Nanoseconds per number: "
      f"lists {per_number['lists, loops']:.1f}, array with loops {per_number['NumPy array, loops']:.1f}, "
      f"vectorized {per_number['NumPy array, vectorized']:.1f}.")

<font color="#c00000" size="5"><b>Your turn.</b></font> The next cell shows one column before and after standardization. Change `COLUMN` to look at other columns, and choose one that your group finds interesting.

In [ ]:
COLUMN = 0      # the position of a column, from 0 to m - 1

name = df.columns[COLUMN]
before, after = X[:, COLUMN], Z[:, COLUMN]
print(f"Dataset {GROUP}, column {COLUMN}: {name}")
print(f"{'':8s}{'minimum':>14s}{'mean':>14s}{'maximum':>14s}{'std. deviation':>16s}")
for label, values in (("before", before), ("after", after)):
    print(f"{label:8s}{values.min():14.4g}{values.mean():14.4g}{values.max():14.4g}{values.std():16.4g}")

fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.4))
left.hist(before, bins=40, color="#2a78d6", edgecolor="white", linewidth=0.6)
left.set_title("Before: the original values")
left.set_xlabel(name)
left.set_ylabel("number of rows")
right.hist(after, bins=40, color="#2a78d6", edgecolor="white", linewidth=0.6)
right.set_title("After: the standardized values")
right.set_xlabel("standard deviations from the mean")
for ax in (left, right):
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Dataset {GROUP}, column {name}")
fig.tight_layout()
plt.show()

<font color="#c00000" size="5"><b>Post now.</b></font> One member posts for the group on the discussion board of this meeting. Begin the post with the names of all the members of your group. The post contains:

- the line for the discussion board that the first cell printed;
- the timing chart, and the two histograms of your column. To save a figure, right-click it and choose **Save image as**;
- under each image, one sentence that says what the image shows;
- one sentence on what surprised your group.

## <font color="#1967d2"><b>Act 2: An image is an array</b></font>

Your image, in colour, is an array of shape (H, W, 3): H rows and W columns of pixels, with three numbers for each pixel. The next cell reads it and turns it to grey with the rule of the slides, in one line on whole channels.

In [ ]:
image = read_image(picture_info["file"])
H, W = image.shape[0], image.shape[1]
grey = 0.299 * image[:, :, 0] + 0.587 * image[:, :, 1] + 0.114 * image[:, :, 2]
print(f"Image {GROUP}: {picture_info['title']} ({picture_info['creator']})")
print(f"shape of image: {image.shape}, which is {image.size:,} numbers; shape of grey: {grey.shape}")
show([image, grey], ["the image", "grey, from the three channels"])

<font color="#c00000" size="5"><b>Your turn.</b></font> Make a new picture from your image by computing with its channels. Complete the function `our_transform` in the next cell, by yourselves or with an AI assistant: exchange two channels, set one channel to 0, square or take the square root of one channel, replace a channel by `grey`, or something of your own. Values below 0 or above 1 are drawn as 0 or 1. Run the cell until the picture is what you want.

In [ ]:
def our_transform(image):
    new = image.copy()               # a copy, so that image stays as it was
    # Your turn: change the array new here. For example: new[:, :, 0] = image[:, :, 2]
    return new


new_picture = our_transform(image)
assert new_picture.shape[:2] == (H, W), "The new picture must have the same height and width as the image."
show([image, new_picture], ["the image", "our transformation"])

<font color="#c00000" size="5"><b>Post now</b></font>, as a reply to your group's first post. One member posts for the group; begin with the names of all its members. The post contains the picture of your image beside its transformation, and one sentence that says what the transformation does to the array.

## <font color="#1967d2"><b>Act 3: A word is a vector</b></font>

The next cell reads the vectors of 50,000 words, with 300 numbers for each word, divides every vector by its length, and defines the two functions of the slides: `nearest`, which finds the words closest to a word, and `analogy`, which answers "a is to b as c is to which word?". It then puts them to your group's word, and to three questions.

In [ ]:
words = urlopen(EMBEDDINGS + "words.txt").read().decode().split()      # the 50,000 words, most frequent first
position = {word: k for k, word in enumerate(words)}                   # the row number of every word
E = np.load(io.BytesIO(urlopen(EMBEDDINGS + "glove-300.npy").read())).astype(np.float32)
U = E / np.linalg.norm(E, axis=1, keepdims=True)                       # every row divided by its length
print("shape of U:", U.shape)


def row(word):
    assert word in position, f"'{word}' is not among the {len(words):,} words. All the words are in lower case."
    return position[word]


def nearest(word, k=8):
    """The k words whose vectors point most nearly in the direction of the vector of word."""
    similarity = U @ U[row(word)]                  # one dot product for every word
    order = np.argsort(-similarity)                # row numbers, from the most similar word on
    return [words[i] for i in order[1:k + 1]]      # order[0] is the word itself


def analogy(a, b, c, k=3):
    """a is to b as c is to which word? The k words closest to: vector of b - vector of a + vector of c."""
    target = U[row(b)] - U[row(a)] + U[row(c)]
    similarity = U @ (target / np.linalg.norm(target))
    similarity[[row(a), row(b), row(c)]] = -2      # the three words of the question cannot be the answer
    return [words[i] for i in np.argsort(-similarity)[:k]]


print(f"closest to '{WORD}':", nearest(WORD))
print("man : king = woman : ?      ", analogy("man", "king", "woman"))
print("france : paris = italy : ?  ", analogy("france", "paris", "italy"))
print("walk : walked = play : ?    ", analogy("walk", "walked", "play"))

<font color="#c00000" size="5"><b>Your turn.</b></font> Replace the three questions in the next cell with three questions of your own, in lower case, and run it. Find one question that the vectors answer well and one that they answer badly.

In [ ]:
print("japan : tokyo = egypt : ?   ", analogy("japan", "tokyo", "egypt"))
print("big : biggest = small : ?   ", analogy("big", "biggest", "small"))
print("mouse : mice = foot : ?     ", analogy("mouse", "mice", "foot"))

<font color="#c00000" size="5"><b>Your turn.</b></font> The next cell draws the vectors of pairs of words in two dimensions, with an arrow from the first word of each pair to the second. Replace the pairs with five to eight pairs of your own that share one relation: a verb and its past form, an animal and its young, a country and its language, or a relation your group thinks of. If the relation is the same vector for every pair, the arrows come out nearly parallel.

In [ ]:
def picture(pairs):
    """Draw the vectors of the words of the pairs in two dimensions, with an arrow from the first word of a pair to the second."""
    chosen = [word for pair in pairs for word in pair]
    V = U[[row(word) for word in chosen]]
    V = V - V.mean(axis=0)
    directions = np.linalg.svd(V, full_matrices=False)[2][:2]     # the two directions of greatest spread
    xy = V @ directions.T
    place = dict(zip(chosen, xy))
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for a, b in pairs:
        ax.annotate("", xy=place[b], xytext=place[a], arrowprops=dict(arrowstyle="->", color="#8a8a86"))
        ax.annotate(a, place[a], textcoords="offset points", xytext=(6, 5))
        ax.annotate(b, place[b], textcoords="offset points", xytext=(6, 5))
    ax.plot([place[a][0] for a, b in pairs], [place[a][1] for a, b in pairs], "o", color="#2a78d6", label="first word of a pair")
    ax.plot([place[b][0] for a, b in pairs], [place[b][1] for a, b in pairs], "s", color="#1a1a19", label="second word of a pair")
    ax.set_title(f"{len(chosen)} word vectors of {U.shape[1]} numbers, drawn in two dimensions")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend()
    fig.tight_layout()
    plt.show()


OUR_PAIRS = [("walk", "walked"), ("play", "played"), ("give", "gave"), ("take", "took"), ("write", "wrote"), ("sing", "sang")]
picture(OUR_PAIRS)

<font color="#c00000" size="5"><b>Post now</b></font>, as a reply to your group's first post. One member posts for the group; begin with the names of all its members. The post contains the picture, with one sentence that says what it shows; the relation your pairs share, and whether the arrows came out nearly parallel; and your best question with its answer.

## After the meeting

- **The full notebooks** of the three acts explain every step and hold the work for after the meeting, with checks that test your code: [Act 1](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-10-act-1.ipynb), [Act 2](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-10-act-2.ipynb), [Act 3](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-10-act-3.ipynb).
- **Review Quiz 10** on Canvas opens when the meeting ends. Its questions name the steps of the full notebooks that they draw on.
- Your copy of this notebook stays in your Google Drive, in the folder *Colab Notebooks*.